In [ ]:
import mne
import os
import matplotlib
import pathlib
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from joblib import Parallel, delayed

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

matplotlib.use("Qt5Agg")
mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

i=0


data_dir = r"C:\Users\admin\Yapay_zeka_projeleri\Deap\sleep_edf_database\sleep-cassette"
folders=os.listdir(data_dir)
dflerin_listesi=[]

while i < len(folders):
    psg_file  = os.path.join(data_dir,folders[i])
    hypno_file= os.path.join(data_dir,folders[i+1])
    
    raw = mne.io.read_raw_edf(psg_file, preload=True)
    
    annotations = mne.read_annotations(hypno_file)
    
    raw.set_annotations(annotations)

    

    event_dict_guncel={
        'Sleep stage W': 0,
        'Sleep stage 1': 1,
        'Sleep stage 2': 2,
        'Sleep stage 3': 3,
        'Sleep stage 4': 3,  
        'Sleep stage R': 4,
        "Sleep stage ?": 5
    }
    
    events, _ =mne.events_from_annotations(raw, event_id=event_dict_guncel ,chunk_duration=30.0)
    
    len(np.where(events[:,2] !=0)[0])
    sleep_indices=np.where(events[:,2] !=0)[0]
    
    
    if len(sleep_indices)>0:
        first_sleep_idx = sleep_indices[0] 
        last_sleep_idx= sleep_indices[-1] 
        
    
        
        start_idx = max(0,first_sleep_idx - 60)
        end_idx   = min(len(events), last_sleep_idx + 60)
    
        
        events_trimmed = events[start_idx:end_idx]
        
    
    else:
        print("Bu hastada hiç uyku bulunamadı!")
        events_trimmed = events
    
    kanal_tipleri= {
        "EOG horizontal": "eog", # göz kanalı
        "EMG submental": "emg", # kas kanalı
        "Resp oro-nasal": "misc", #misc(miscellaneous)=Diğer/ıvır zıvır
        "Temp rectal": "misc",
        "Event marker": "stim" # uyaran/olay kanalı   
    }

    raw.set_channel_types(kanal_tipleri)
    
    raw_eeg_filtered=raw.filter(l_freq=0.1,h_freq=40)
    
    

    tmin = 0
    tmax = 30
    baseline = None
    
    epochs = mne.Epochs(raw_eeg_filtered,
                        events=events_trimmed,
                        event_id=event_dict_guncel,
                        tmin=tmin,
                        tmax=tmax,
                        baseline=baseline,
                        preload=True,
                       on_missing='warn')
    
  
    
    reject_criteria = dict(eeg=350e-6,      
                           eog=573e-6)       
    
    flat_criteria = dict(eeg=1e-6)           
    
    epochs=epochs.drop_bad(reject=reject_criteria, flat=flat_criteria)
    
    bdf = pd.DataFrame({'Sleep_Stage': epochs.events[:, 2]})
        
    
    
    df = pd.DataFrame({'Sleep_Stage': epochs.events[:, 2]})
    
    psd_obj = epochs.compute_psd(method='welch', fmin=0.5, fmax=30.0, picks='eeg', verbose=False)
    psds, freqs = psd_obj.get_data(return_freqs=True) 
    
    rel_psds = psds / psds.sum(axis=2, keepdims=True)
    
    bands = {'Delta': (0.5, 4), 'Theta': (4, 8), 'Alpha': (8, 12), 'Sigma': (12, 16), 'Beta': (16, 30)}
    
    for ch_idx, ch_name in enumerate(psd_obj.ch_names):
        for band, (fmin, fmax) in bands.items():
            freq_mask = (freqs >= fmin) & (freqs < fmax)
            df[f"{ch_name}_{band}_Rel"] = rel_psds[:, ch_idx, freq_mask].sum(axis=1)
    
    data = epochs.get_data() 
    for ch_idx, ch_name in enumerate(epochs.ch_names):
        df[f"{ch_name}_Variance"] = np.var(data[:, ch_idx, :], axis=1) * 1e12


    dflerin_listesi.append(df)
    i+=2
    if i %2 ==0:
        print(f"Dataframe hazırlama süreci yüzdelik ilerleme:%{(int(i)/int(len(folders))) * 100}")
birlesmis_df = pd.concat(dflerin_listesi, ignore_index=True,axis=0)


print("Dataframe eğitiliyor")
y=birlesmis_df["Sleep_Stage"].to_numpy()
X=birlesmis_df.drop("Sleep_Stage",axis=1)

all_data_pipeline=Pipeline(steps=[
    ("eksik_deger_int",SimpleImputer(strategy="mean")),
    ("standardize",StandardScaler())
])

X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2,random_state=42)



classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=1),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_jobs=1), 
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(n_jobs=1)
}

def model_egit_ve_test_et(name, model, preprocessor, X_train, y_train, X_test, y_test):
    
    current_pipe = Pipeline(steps=[
        ('hazirlik', all_data_pipeline),
        ('model', model)
    ])
    
    current_pipe.fit(X_train, y_train)
    
    y_pred = current_pipe.predict(X_test)
    
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred, average="weighted"),
        "Recall": recall_score(y_test, y_pred, average="weighted"),
        "Precision": precision_score(y_test, y_pred, average="weighted"),
        "Trained_Model": current_pipe
    }

print("Modeller paralel olarak eğitiliyor... Lütfen bekleyin.")


yapilacak_isler_listesi = []

for name, model in classifiers.items():
    

    gorev_paketi = delayed(model_egit_ve_test_et)(
        name, 
        model, 
        all_data_pipeline, 
        X_train, 
        y_train, 
        X_test, 
        y_test
    )
    yapilacak_isler_listesi.append(gorev_paketi)


""" yukarıdaki for döngüsü aşağıdaki gibi kısaca yazılabilir

sonuclar = Parallel(n_jobs=-1)(
    delayed(model_egit_ve_test_et)(
        name, model, filtrelerim, X_train, y_train, X_test, y_test
    ) for name, model in classifiers.items()
)

"""

motor = Parallel(n_jobs=-1, verbose=10)

print("Modeller eğitiliyor...")
sonuclar = motor(yapilacak_isler_listesi)

tum_modeller = {}
skor_listesi = []

for sonuc in sonuclar:
    model_adi = sonuc["Model"]
    tum_modeller[model_adi] = sonuc["Trained_Model"]
    
    skor_listesi.append({
        k: v for k, v in sonuc.items() if k != "Trained_Model"
    })


df_sonuc = pd.DataFrame(sonuclar)
df_sonuc = df_sonuc.sort_values(by="F1 Score", ascending=False)